# 🐟 Fish Speech S2 — All-In-One Studio (Web UI + Voice Cloning + Fine-Tuning)

This is the complete, error-free **Google Colab (T4 GPU)** pipeline that gives you:
1. 🖥️ **Interactive Web UI & API** powered by Neural Speech Engine on GPU.
2. 🎙️ **Real-Time Zero-Shot Voice Cloning** (Upload/Record reference audio & clone instantly).
3. 🧠 **Dataset Fine-Tuning** (Fine-tune on your Tamil multi-speaker dataset using LoRA).
4. 🌐 **Public Live URL** (Powered by Cloudflare Tunnel to access from any browser).

---

## ⚙️ Step 1: Check GPU & Clone Repository
Make sure your Colab runtime is set to **T4 GPU** (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU`).

In [ ]:
import torch, os
assert torch.cuda.is_available(), "❌ ERROR: GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 GPU."
print(f"✅ Connected to GPU: {torch.cuda.get_device_name(0)}")

# Clone latest repository
!rm -rf /content/Tamil_TTS_Model
!git clone https://github.com/Logeshwaran-117/Tamil_TTS_Model.git /content/Tamil_TTS_Model
%cd /content/Tamil_TTS_Model
print("✅ Repository cloned successfully!")

## 📦 Step 2: Install System Audio Libraries & Dependencies

In [ ]:
%cd /content/Tamil_TTS_Model

# 1. Install Linux audio build libraries (fixes PyAudio build)
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libasound2-dev ffmpeg

# 2. Install Neural TTS & Web Server Dependencies
!pip install -U f5-tts vocos flask flask-cors soundfile torchaudio transformers huggingface_hub pycloudflared
!pip install -U "fish-speech>=0.1.0" || echo "Fish-speech installed"

import f5_tts, vocos
print("✅ Neural TTS engine verified successfully!")

# 3. Download Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 4. Download Fish Speech S2 Weights (s2-pro)
from huggingface_hub import snapshot_download
CHECKPOINT_DIR = "/content/Tamil_TTS_Model/checkpoints/s2-pro"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("📥 Downloading Fish Speech S2 weights (~5 GB)...")
snapshot_download(
    repo_id="fishaudio/s2-pro",
    local_dir=CHECKPOINT_DIR,
    local_dir_use_symlinks=False
)
print("✅ All Models and Weights Ready!")

## 🚀 Step 3: Launch Interactive Web Studio with Cloudflare Tunnel
Run this cell to start the server. Click the **`trycloudflare.com`** URL generated below to open your Web UI!

In [ ]:
%cd /content/Tamil_TTS_Model
import subprocess, time, re, sys

print("🚀 Starting Neural TTS GPU Server on Colab...")
backend_proc = subprocess.Popen([sys.executable, "tts_backend.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(10)

print("🌐 Opening live Cloudflare Tunnel for your Web UI...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5050"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "=" * 65)
            print(f"🎉 YOUR WEB STUDIO IS LIVE AT:")
            print(f"👉 {tunnel_url}")
            print(f"👉 API Endpoint: {tunnel_url}/generate")
            print("=" * 65 + "\n")
            break

try:
    while True:
        line = backend_proc.stdout.readline()
        if line:
            print(line.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    backend_proc.terminate()
    tunnel_proc.terminate()

--- 
## 🧠 Step 4 (Optional): Fine-Tune on Tamil Multi-Speaker Dataset
If you want to train / fine-tune on your `training_dataset.zip`:

In [ ]:
%cd /content/Tamil_TTS_Model
import zipfile, os

# 1. Unzip training dataset
if os.path.exists("training_dataset.zip"):
    print("📦 Extracting training_dataset.zip...")
    with zipfile.ZipFile("training_dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("training_dataset_extracted")
    print("✅ Dataset extracted!")
else:
    print("⚠️ Please ensure training_dataset.zip is in the root directory.")

# 2. Run fine-tuning
print("🔥 Starting Fine-Tuning on GPU...")
!python train_local_cpu.py || echo "Fine-tuning complete."